# Tune late-delivery classification on Azure Machine Learning

**What happens:** this notebook prepares chronological Olist train/validation data, registers two MLTable assets, and submits an Azure ML **sweep job**. The notebook is local; each training trial runs on Azure's `olist-rai-cpu` cluster. The June-August **test data never enters the sweep**.

**Limits:** at most 8 trials, 1 at a time, 30 minutes per trial, and **300 minutes total** (`300 * 60 = 18,000` seconds). The overall timeout limits the sweep job, not every Azure storage or workspace charge. Registration and submission are opt-in; an ignored local file retains the submitted sweep ID.

This is a separate tuning exercise. It does not replace the registered `olist-late-delivery-rai:1` dashboard model or retune it on the held-out test set. [Official sweep-job guidance](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-tune-hyperparameters?view=azureml-api-2).

## 1. Select the Python environment and lab folder

From the cloned repository root, run `python scripts/bootstrap.py --subscription ... --resource-group ... --workspace ... --create-cluster` if you need the scale-to-zero cluster (see root README). It extracts the shared CSVs, writes ignored `config.json`, and registers the root `.venv` kernel **Python (Olist Azure ML)** (`azure-ai-ml==1.35.1`). Select that kernel and open this notebook inside the exercise folder. Preparation runs locally; fitting happens only after an explicitly enabled cloud submission.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

marker = Path('datasets/olist/Ecommerce Dataset.zip')
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / marker).is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook inside the cloned repo containing datasets/olist/Ecommerce Dataset.zip')
LAB = ROOT / 'exercises' / '07-late-delivery-sweep'
RAW = ROOT / 'datasets' / 'olist' / 'Ecommerce Dataset'
if not (LAB / 'prepare.py').is_file() or not (RAW / 'olist_orders_dataset.csv').is_file():
    raise FileNotFoundError('Run scripts/bootstrap.py from the repo root to extract the shared Olist CSVs')
config = json.loads((ROOT / 'config.json').read_text(encoding='utf-8'))
REGISTER_RESOURCES = False  # Opt in to upload missing assets and register environment v2.
SUBMIT_SWEEP = False  # Separately opt in to a billable sweep after reviewing the limits.
print('Lab:', LAB.resolve())
print('Kaggle CSV folder:', RAW.resolve())

## 2. Prepare training and validation, without looking at test labels

The same order-level joins and date-corrected late label as the RAI lab are reused. This time the split is **before April 2018** for training and **April-May 2018** for validation. Missing distance is filled from the earlier training split only. The June-August test asset is not read or uploaded by this notebook.

In [ ]:
subprocess.run([sys.executable, str(LAB / 'prepare.py'), '--sweep-only',
                '--raw-data', str(RAW)], check=True)
for split in ['sweep-train', 'sweep-validation']:
    frame = pd.read_parquet(LAB / 'data' / split / 'data.parquet')
    print(split, 'orders:', len(frame), 'late:', int(frame.late_delivery.sum()),
          'late rate:', f'{frame.late_delivery.mean():.1%}')
    assert frame.notna().all().all()
    assert 'order_id' not in frame and 'order_delivered_customer_date' not in frame

## 3. Connect to your Azure ML workspace

The nonsecret resource IDs come from the root ignored `config.json` written by bootstrap. Sign in with `az login` in a terminal. The `olist-rai-cpu` cluster must already exist (bootstrap creates it only with `--create-cluster`). No workspace or compute is created by this notebook.

In [ ]:
from azure.ai.ml import MLClient, Input, command
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data
from azure.ai.ml.sweep import Choice
from azure.identity import AzureCliCredential

SUBSCRIPTION_ID = config['subscription_id']
RESOURCE_GROUP = config['resource_group']
WORKSPACE = config['workspace_name']
ml_client = MLClient(AzureCliCredential(), SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE)
print('Workspace:', ml_client.workspaces.get(WORKSPACE).name)
print('Compute:', ml_client.compute.get('olist-rai-cpu').name)

## 4. Register two versioned MLTable assets

**Opt-in cloud action:** set `REGISTER_RESOURCES = True` in step 1 to upload missing Parquet-backed MLTable assets and register the environment. Existing version 1 assets are reused, never overwritten. To publish changed data use a new version and update sweep inputs.

In [ ]:
from azure.core.exceptions import ResourceNotFoundError

for split in ['sweep-train', 'sweep-validation']:
    name = f'olist-late-{split}'
    try:
        asset = ml_client.data.get(name=name, version='1')
        print('Reusing:', asset.name, asset.version)
    except ResourceNotFoundError:
        if not REGISTER_RESOURCES:
            print('Missing:', name, 'version 1; enable REGISTER_RESOURCES to upload')
        else:
            asset = ml_client.data.create_or_update(Data(
                name=name, version='1', type=AssetTypes.MLTABLE,
                path=str(LAB / 'data' / split),
            ))
            print('Registered:', asset.name, asset.version)

## 5. Use a workspace-owned environment for sweep trials

Azure ML's sweep backend cannot use the external Docker build-context URL of the registry `responsibleai-tabular` environment used by ordinary jobs. The small workspace environment in `environment/` builds from a published Azure ML base image and installs scikit-learn, PyArrow and MLflow. Register it once; Azure caches the image after its first build.

In [ ]:
from azure.ai.ml.entities import Environment
try:
    sweep_environment = ml_client.environments.get(name='olist-sweep-env', version='2')
    print('Reusing environment:', sweep_environment.name, sweep_environment.version)
except ResourceNotFoundError:
    if not REGISTER_RESOURCES:
        print('Missing olist-sweep-env:2; enable REGISTER_RESOURCES to register it')
    else:
        sweep_environment = ml_client.environments.create_or_update(Environment(
            name='olist-sweep-env', version='2',
            image='mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04',
            conda_file=str(LAB / 'environment' / 'conda.yml'),
        ))
        print('Registered workspace environment:', sweep_environment.name, sweep_environment.version)

## 6. Describe one cloud training trial

Each trial runs `src/sweep_train.py` in Azure ML's `olist-sweep-env:2` environment. The trial tries a logistic-regression regularization value `c` and a late-class weight. It logs **validation_average_precision** from positive-class probabilities; precision and recall are logged for interpretation. We do not optimize classification accuracy on this imbalanced dataset.

In [ ]:
base_trial = command(
    code=str(LAB / 'src'),
    command=('python sweep_train.py --train ${{inputs.training_data}} '
             '--validation ${{inputs.validation_data}} --c ${{search_space.c}} '
             '--positive-weight ${{search_space.positive_weight}}'),
    environment='azureml:olist-sweep-env:2',
    inputs={
        'training_data': Input(type=AssetTypes.MLTABLE, path='azureml:olist-late-sweep-train:1'),
        'validation_data': Input(type=AssetTypes.MLTABLE, path='azureml:olist-late-sweep-validation:1'),
    },
)
search_space = {
    'c': Choice(values=[0.01, 0.1, 1.0, 10.0]),
    'positive_weight': Choice(values=[1.0, 3.0, 5.0, 10.0]),
}
print('Search: 4 regularization choices x 4 positive-class weights')

## 7. Limit the entire sweep to 300 minutes

Azure ML's SDK expects **seconds**: `timeout=300*60` limits the whole sweep to five hours; `trial_timeout=30*60` limits one trial to half an hour. We additionally cap the search at eight trials, with only one running at a time because the existing cluster has one node. If Azure ML stops early, the cap is still respected. The `primary_metric` must match the MLflow metric name in `sweep_train.py` exactly.

In [ ]:
TOTAL_TIMEOUT_MINUTES = 300
TRIAL_TIMEOUT_MINUTES = 30
MAX_TOTAL_TRIALS = 8
MAX_CONCURRENT_TRIALS = 1
sweep_job = base_trial.sweep(
    compute='olist-rai-cpu', sampling_algorithm='random',
    search_space=search_space,
    primary_metric='validation_average_precision', goal='maximize',
    max_total_trials=MAX_TOTAL_TRIALS,
    max_concurrent_trials=MAX_CONCURRENT_TRIALS,
    timeout=TOTAL_TIMEOUT_MINUTES * 60,
    trial_timeout=TRIAL_TIMEOUT_MINUTES * 60,
)
sweep_job.experiment_name = 'olist-late-delivery-sweep'
sweep_job.display_name = 'olist-late-ap-sweep-300min'
print('Azure ML sweep: at most', MAX_TOTAL_TRIALS, 'trials;',
      TOTAL_TIMEOUT_MINUTES, 'minutes total;',
      TRIAL_TIMEOUT_MINUTES, 'minutes per trial')

## 8. Submit ONE billable Azure ML sweep

The submission cell uploads trial code and launches training **only if** `SUBMIT_SWEEP = True` in step 1. The returned job name is persisted in ignored `.sweep-run-id.json` beside this notebook; to reconnect to a previously submitted job in your workspace, put its name in that file as JSON with a `job_name` key. Reruns reconnect instead of submitting another sweep. The 300-minute limit is not an absolute cloud spending limit for storage or the workspace.

In [ ]:
run_id_file = LAB / '.sweep-run-id.json'
existing_name = json.loads(run_id_file.read_text(encoding='utf-8'))['job_name'] if run_id_file.is_file() else None
if existing_name:
    submitted = ml_client.jobs.get(existing_name)
elif SUBMIT_SWEEP:
    submitted = ml_client.jobs.create_or_update(sweep_job)
    run_id_file.write_text(json.dumps({'job_name': submitted.name}) + '\n', encoding='utf-8')
else:
    submitted = None
    print('Sweep not submitted; set SUBMIT_SWEEP = True only when ready to incur charges')
if submitted:
    print('Sweep job name:', submitted.name)
    print('Azure ML Studio:', submitted.studio_url)

## 9. Inspect progress and the best trial

This is read-only; rerun it later rather than creating another job. Azure ML Studio shows child trials and their metrics under the sweep. After `Completed`, inspect `best_child_run_id`, its hyperparameters, and its validation average precision. **Do not select on average precision alone for an alert system:** in the completed course run the highest-AP trial (0.1623) predicted no late orders at its default threshold (precision and recall both 0). A separately tested fixed setting (`C=1`, positive weight `3`) achieved AP 0.1632, precision 0.2717, and recall 0.1335 on the same validation period. Neither is ready for automatic alerts. The June-August test remains unused by this sweep.

In [ ]:
if submitted is None:
    print('No sweep job ID to inspect yet')
else:
    current = ml_client.jobs.get(submitted.name)
    print('Sweep status:', current.status)
    for child in ml_client.jobs.list(parent_job_name=submitted.name):
        print(child.name, child.status)
if submitted is not None and current.status == 'Completed':
    winner_name = (current.properties or {}).get('best_child_run_id')
    if winner_name:
        winner = ml_client.jobs.get(winner_name)
        parameters = json.loads((winner.tags or {}).get('hyperparameters', '{}'))
        print('Best sampled child:', winner.name, 'parameters:', parameters,
              'validation AP:', (current.properties or {}).get('score'))
        print('Check late precision/recall in Studio; compare the fixed C=1, weight=3 trial.')
    else:
        print('Open the completed sweep in Studio to identify the highest AP trial.')
elif submitted is not None and current.status in ('Failed', 'Canceled'):
    print('The sweep did not finish successfully. Inspect the child jobs before starting a new paid run.')
elif submitted is not None:
    print('Leave the June-August test unused while the sweep is running.')

## What this exercise does not do

A sweep selects hyperparameters on **April-May validation**, not the June-August test. Do not repeatedly test candidates against future data. Random search tried only 8 of 16 combinations and did not outperform the fixed AP benchmark. If an alert is needed, define a validation-time decision threshold and minimum acceptable late recall and precision, then retrain once on all pre-June delivered orders and run one final future-period evaluation. The registered RAI model from the earlier lab remains a separate demonstration. `class_weight` can change false-alert counts, so never treat the highest AP as deployment approval.

**Sources:** [Azure ML sweep documentation](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-tune-hyperparameters?view=azureml-api-2), [SDK sweep-job limits](https://learn.microsoft.com/en-us/python/api/azure-ai-ml/azure.ai.ml.sweep.sweepjob?view=azure-python), [MLflow metrics in Azure ML](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-log-view-metrics?view=azureml-api-2).